# **EAS 4860/6860: Moist Convection**

In this Jupyter notebook, you will use soundings to understand moist convective stability. You will need to run this notebook on the EAS student server, where the specialized meteorological libraries (like cartopy, metpy, and eccodes) are already optimized for our use. Please use the "EAS 4860: Tropical Meteorology and Climate" kernel when running this notebook. We will use this throughout the semester. Run the below cell to import the necessary packages.

In [9]:
import numpy as np
import os
import sys
import matplotlib.pyplot as plt

# Get the absolute path to your helpers folder
helper_path = os.path.abspath('./helpers')

# Add it to the system path if it's not already there
if helper_path not in sys.path:
    sys.path.append(helper_path)
print(helper_path)

# Add the directory that CONTAINS the helpers folder
parent_path = os.path.abspath('.')
sys.path.append(parent_path)

from helpers import thermo

from helpers import thermo

ModuleNotFoundError: No module named 'thermo'

## **Sounding Data**
Let us plot the classical skew-$T_\rho$-log-$p$ chart for some tropical island locations. The first island we will visit is Majuro, the capital of the Marshall Islands. The below code download sounding data from the specified station, on a date that you specify through the variables `year`, `month`, `day`, and `hour`.

In [ ]:
# Acquire station ID of the Majuro station.
station_name = 'Majuro'
station_id = thermo.get_station_id(station_name)

# Download sounding from the below date
year, month, day, hour = (2018, 11, 1, 0)
data, header, status = thermo.getsounding(station_id, year, month, day, hour)
if status == 0:
    print('No data for this station and time')

# Pressure, temperature, and relative humidity from sounding data.
pz = data[:, 0]                 # pressure, hPa
tz = data[:, 2] + 273.15        # temperature, K
rhz = data[:, 4] / 100          # relative humidity, percentage
qz = 0.001 * data[:, 5]         # specific humidity, kg/kg

# Remove all pressure levels with missing data (NaN)
valid_mask = (~np.isnan(pz)) & (~np.isnan(tz)) & (~np.isnan(rhz)) & (~np.isnan(qz))
# Ensure unique pressure levels
p_unique_mask = np.insert(np.diff(pz) != 0, 0, True)
final_mask = valid_mask & p_unique_mask
final_mask[0] = False           # remove the first level

# Sounding data
p = pz[final_mask]              # pressure, hPa
T = tz[final_mask]              # temperature, K
q = qz[final_mask]              # specific humidity, kg/kg
rh = rhz[final_mask]            # relative humidity, percentage
n = len(p)                      # number of levels

We can easily plot the temperature and humidity of the sounding data using the `skewtv` method below.

In [ ]:
T_C = T - 273.15                # temperature, C
success = thermo.skewtv(p, T_C, rh)
titstring = f"Station {station_id}, {station_name}, {year}/{month}/{day}/{hour}"
plt.title(titstring)

## **Lifting a Moist Parcel**
Next, let us consider lifting a parcel from a specified pressure level. This pressure level is specified by `p_input`. The following code finds the pressure level closest to your requested pressure level.

In [ ]:
# Pressure to lift parcel reversibly from
p_input = 1000                   # hPa

# Find the data level closest to the requested pressure
ibest = np.argmin(np.abs(p - p_input))

# Obtain the pressure, temperature, and humidity of the parcel at the
# requested pressure level
p0 = p[ibest]                   # pressure
T0 = T[ibest]                   # temperature
q0 = q[ibest]                   # specific humidity

# Check for NaNs and try the next level up if necessary
if np.isnan(T0) or np.isnan(q0):
    ibest += 1
    p0 = p[ibest]
    if p0 < 500:
        print("Could not find viable parcel level")
    T0 = T[ibest]
    q0 = q[ibest]

This section simulates the lifting of an air parcel from its starting state $(T_0, q_0, p_0)$ through the atmospheric column, starting from your specified pressure level. It calculates the trajectory for two different thermodynamic assumptions: reversible moist adiabatic and pseudoadiabatic. In a reversible process, all condensed water remains within the parcel. The total water mass is conserved, contributing to the parcel's weight and affecting its buoyancy. In a pseudoadiabatic process, all condensed water is assumed to precipitate out of the parcel immediately upon formation. Only the latent heat remains to warm the parcel (and increase its buoyancy).

In class, we found that the density temperature allows us to determine the buoyancy of a moist parcel. The density temperature is defined as:
$$T_{\rho} = T \frac{1 + r/\epsilon}{1 + r_t}$$
where $r$ is the mixing ratio, and $r_t$ is the total water mixing ratio.

In [ ]:
# Define all pressure levels through which to lift the parcel.
plift = p[ibest:]
Rd = 287.05         # J / kg K
Rv = 461.5          # J / kg K
eps = Rd / Rv

def density_temp(T0, q0, p0, plift):
    # Calculate the density temperature of a parcel with a temperature
    # of T0, humidity of q0, and pressure of p0, through an atmosphere
    # defined by the pressure levels plift. Lift the parcel both
    # reversibly and pseudo-adiabatically.

    # Calculate the temperature and humidity by lifting the
    # parcel along a reversible moist adiabat.
    T_lift_rev, qs_lift_rev = thermo.svinvert(T0, q0, p0, plift)
    qs_lift_rev = np.minimum(q0, qs_lift_rev)
    # Calculate the density temperature of this parcel,
    # and convert it to degrees K.
    Trho_lift_rev = T_lift_rev * (1 + qs_lift_rev / eps) / (1 + q0)

    # Calculate the temperature and humidity by lifting the
    # parcel pseudoadiabatically.
    T_lift_pa, qs_lift_pa = thermo.svinvertpa(T0, q0, p0, plift)
    qs_lift_pa = np.minimum(q0, qs_lift_pa)
    # Calculate the density temperature of the parcel,
    # that is lifted pseudoadiabatically.
    Trho_lift_pa = T_lift_pa * (1 + qs_lift_pa / eps) / (1 + qs_lift_pa)
    return Trho_lift_rev, Trho_lift_pa

Trho_lift_rev, Trho_lift_pa = density_temp(T0, q0, p0, plift)

After computing the density temperature of a moist parcel lifted both reversibly and pseudoadiabtically, let us plot the temperatures alongside the environmental density temperature, in order to assess the buoyancy of the moist parcel with respect to the environment.

In [ ]:
# Plot the skew-T log p diagram
thermo.skewtv(p, T_C, rh);
titstring = f"Station {station_id}, {station_name}, {year}/{month}/{day}/{hour}"

# Plot the density temperature of a parcel lifted both reversibly
# and pseudo-adiabatically, alongside the environmental fields.
# Use a log-pressure adjustment (-30 * log(p)) to transform
# temperatures into Skew-T X-axis coordinates.
Tz = (Trho_lift_rev - 273.15) - 30.0 * np.log(0.001 * plift)
Tzpa = (Trho_lift_pa - 273.15) - 30.0 * np.log(0.001 * plift)
g1, = plt.plot(Tz, plift, 'b--', linewidth=2., label='Reversible')
g2, = plt.plot(Tzpa, plift, 'm--', linewidth=2., label='Pseudo-adiabatic')

plt.legend(handles=[g1, g2])
plt.title(f"{titstring}; Parcel lifted from {plift[0]:.1f} hPa")
plt.show()

**Key Questions to Consider:**
1. Describe the shape of the temperature profile of an ascending moist parcel, under both reversible and pseudo-adiabatic ascent.
2. Compare the density temperature of a parcel in reversible vs. pseudo-adiabatic ascent. What are the differences, and why do you think this is the case?
3. What is the buoyancy of an ascending moist parcel compared to the environment, for both reversible and pseudo-adiabatic ascent?

# **Quantifying Buoyancy of a Parcel**
We can quantify the buoyancy of a parcel lifted from our requested pressure level, using the density temperature:
$$
    B = g \frac{T_{\rho, p} - \overline{T}_{\rho}}{\overline{T}_{\rho}}
$$
where $\overline{T}_{\rho}$ is the density temperature of the environment, and $T_{\rho, p}$ is the density temperature of the parcel that is lifted. Remember that implicit in the density temperature calculation is a closure for how $r_t$ evolves as the parcel is lifted!

Let us plot the difference between the density temperature of the parcel and the environment. The sign of this is proportional to buoyancy.

In [ ]:
# Calculate the density temperature of the environmental field.
Trho_env = T * (1 + q / 0.622) / (1 + q)         # density temperature, K

buoyancy_rev = Trho_lift_rev - Trho_env[ibest:]
buoyancy_pa = Trho_lift_pa - Trho_env[ibest:]

# Plot the difference in density temperature between a reversibly/pseudo-adiabatically
# lifted parcel and the density temperature of the environment.
plt.figure()
plt.plot(buoyancy_rev, plift, 'b', label = 'Reversible')
plt.plot(buoyancy_pa, plift, 'm', label = 'Pseudo-Adiabatic')
plt.ylim([1000, 100])
plt.xlim([-6, 6])
plt.grid()
plt.xlabel('Buoyancy (K)')
plt.ylabel('Pressure (hPa)')
plt.legend()

**Key Question to Consider:** What is the buoyancy of an ascending moist parcel compared to the environment, for both reversible and pseudo-adiabatic ascent? When lifted from the starting pressure level, would you expect the moist parcel to ascend? Which level would it ascend to?

In the above, we picked a particular pressure level, `plift`, at which we began lifting a moist parcel. Since the buoyancy of a parcel depends on its initial state, the buoyancy profile can look much different for if we picked a different initial pressure level.

**Key Question to Consider:** Change the pressure level at which the parcel is lifted. Does the buoyancy profile differ? Experiment with different pressure levels!

# **Diagrams of Parcel Buoyancy**
As you can see above, the buoyancy of a moist parcel depends heavily on which pressure level it is lifted from. To easily compare them, we want to calculate the buoyancy of parcels lifted from many different origin levels. The code below does this.

In [ ]:
# Matrices storing the buoyancy of a parcel lifted reversibly and
# pseudoadiabatically.
Brev = np.zeros((n, n))
Bpseudo = np.zeros((n, n))

# Calculate the buoyancy of all parcels lifted from the surface to the
# minimum parcel origin pressure (pmin).
pmin=850;       # Minimum parcel origin pressure (hPa)
for i in range(n):
    if p[i] >= pmin:
        # Define levels above the parcel origin
        plift = p[i+1:]

        # Conduct both reversible and pseudo-adiabatic lifting
        # to calculate the density temperature.
        Trho_lift_rev, Trho_lift_pa = density_temp(T[i], q[i], p[i], plift)
        Brev[i, i+1:] = Trho_lift_rev - Trho_env[i+1:]
        Bpseudo[i, i+1:] = Trho_lift_pa - Trho_env[i+1:]

def plot_buoyancy(p, B):
    ptop=80;        # Minimum pressure (hPa) for graphical display
    Bmin=-15;       # Minimum buoyancy (C) to plot in graphs
    Yaxis='log';    # Scaling of pressure axis for plots ('linear' or 'log')
    ptop = max(ptop, p[-1])

    # Plot the buoyancy
    fig, ax = plt.subplots(figsize=(6, 4.5))

    # Define the standard pressure levels for ticks
    p_ticks = [1000, 900, 800, 700, 600, 500, 400, 300, 200, 100]
    p_labels = [str(x) for x in p_ticks]
    cf = ax.contourf(p, p, B.T, cmap='RdBu_r', levels=np.arange(Bmin, -Bmin+.01, 1), extend='both')

    # Add the zero buoyancy line
    ax.contour(p, p, B.T, levels=[0], colors='k', linewidths=1.5)

    # Formatting
    fig.colorbar(cf, ax=ax, label='Buoyancy (K)')
    ax.set_xscale('linear')
    ax.set_yscale('log' if Yaxis == 'log' else 'linear')
    ax.set_yticks(p_ticks)
    ax.set_yticklabels(p_labels)
    ax.invert_xaxis()
    ax.set_xlim(p[0], pmin)
    ax.invert_yaxis()
    ax.set_ylim([p[0], 100])
    ax.set_xlabel('Parcel Origin Pressure (hPa)')
    ax.set_ylabel('Pressure (hPa)')
    ax.set_title('Buoyancy (K)')
    plt.tight_layout()
    plt.show()

plot_buoyancy(p, Brev)
plot_buoyancy(p, Bpseudo)

**Key Questions to Consider:**
1. How does the buoyancy profile of a moist parcel change as you modify the parcel origin pressure?
2. How does the level of free convection (LFC) shift as you vary the initial pressure?
3. Which parcels would you most likely see ascend in the atmospheric profile?

## **Convective Available Potential Energy and Convective Inhibition**

How much work can be done by a parcel lifted from a origin level $i$ to its level of neutral buoyancy? The level of neutral buoyancy is defined as the highest level after which the buoyancy vanishes. This is defined by the convective available potential energy (CAPE):
$$
\text{CAPE}_i = \int_i^{\text{LNB}} B dz
$$
CAPE has units of $m^2/s^2$, which is an energy. Note that CAPE has to be defined with respect to a parcel being lifted from some origin level $i$. Since sounding data is typically given on pressure levels, it is easier to calculate CAPE in log-pressure coordinates:
$$
    \text{CAPE}_i = \int_{\text{LNB}}^i R_d (T_{\rho, p} - \overline{T}_\rho) d \ln p
$$

We can also ask a related question: how much energy does a parcel need to overcome any negative buoyancy? This is defined by the convective inhibition:
$$
    \text{CIN}_i = - \int^i_{\text{LNB}} R_d \min(0, T_{\rho, p} - \overline{T}_\rho) d \ln p
$$
where we only include non-positive values of buoyancy in the calculation of CIN. Again, CIN has units of $m^2/s^2$.

The below functions will calculate CAPE and CIN given the buoyancy matrices you calculated earlier.

In [ ]:
def calculate_CAPE_CIN(p, B, n):
    CAPE = np.zeros(n)  # CAPE
    CIN = np.zeros(n)   # CIN
    # Calculate log pressure intervals
    # lpi(j) corresponds to 0.5 * log(p_j * p_{j+1})
    lpi = 0.5 * np.log(p[:-1] * p[1:])

    # Loop through parcel origin levels
    for i in range(n - 1):
        # Only calculate CAPE if a parcel has positive buoyancy
        # somewhere along its ascent in the atmosphere.
        if np.max(B[i, :]) > 0:
            # The LNB is the highest level in the atmosphere that
            # has positive buoyancy.
            # np.where returns indices where the mask is True
            i_lnb = i + np.where(B[i, i:-1] > 0)[0][-1]

            # Integrate from parcel origin level to LNB
            for j in range(i + 1, i_lnb + 1):
                lndp = (lpi[j-1] - lpi[j])
                CAPE[i] += Rd * B[i, j] * lndp
                CIN[i] -= Rd * min(B[i, j], 0) * lndp
        else:
            CAPE[i] = np.nan
            CIN[i] = np.nan

    # CIN is only defined when CAPE is positive.
    CIN[CAPE < 0] = 0

    # Ensure only positive values of CAPE.
    CAPE = np.maximum(CAPE, 0)
    CIN = np.maximum(CIN, 0)

    return CAPE, CIN

Finally, we can plot CAPE and CIN as a functin of parcel origin pressure.

In [ ]:
CAPEr, CINr = calculate_CAPE_CIN(p, Brev, n)
CAPEp, CINp = calculate_CAPE_CIN(p, Bpseudo, n)

def plot_CAPE_CIN(p, CAPE, CIN):
    """
    Visualizes CAPE and CIN as a function of the parcel's starting pressure.
    Note: CIN is scaled by 10 to make it visible on the same axis as CAPE.
    """
    fig, ax = plt.subplots(figsize=(6, 4.5))

    ax.plot(p, CAPE, 'b-', linewidth=2, label='CAPE')

    # CIN multiplied by 10 for visibility
    ax.plot(p, 10 * CIN, 'r-', linewidth=1.5, label='10 $\\times$ CIN')
    ax.invert_xaxis()
    ax.set_xlim(p[0], pmin)

    # Formatting and labeling
    ax.set_xlabel('Parcel Origin Pressure (hPa)')
    ax.set_ylabel('Energy (J/kg)')
    ax.set_ylim([0, None])
    ax.grid(True, linestyle=':', alpha=0.9)
    ax.legend(frameon=True, loc='best')

    plt.tight_layout()
    plt.show()

plot_CAPE_CIN(p, CAPEr, CINr)
plot_CAPE_CIN(p, CAPEp, CINp)

**Key Questions to Consider:**
1. Describe the CAPE and CIN of a parcel lifted from varying origin pressures. How do they change as you change the origin pressure?
2. How does the inclusion of water loading change both CAPE and CIN? Are the differences large or small?

# **Potential Intensity**

Potential intensity, the theoretical maximum wind speed of a steady-state, axisymmetric tropical cyclone, can be formulated as:
$$
    V_p^2 = \frac{T_s}{T_o} \frac{C_k}{C_d} (\text{CAPE}^* - \text{CAPE}_b)
$$
where $\text{CAPE}^*$ is the CAPE of a *saturated* parcel lifted from $T_s$ at the tropical cyclone eyewall pressure, while $\text{CAPE}_b$ is the CAPE of a parcel lifted from the boundary layer, with an initial temperature and humidity of the surface air.

Let us calculate the potential intensity using the above algorithms.

In [ ]:
# Lift the parcel from the first sounding level
ibest = 0
plift = p[ibest:]
p0 = p[ibest]                   # pressure
T0 = T[ibest]                   # temperature
q0 = q[ibest]                   # specific humidity

# Calculate the density temperature of a lifted saturated parcel
# at the temperature of an assumed SST. Note, we do NOT include
# the pressure dependent effect of the eyewall.
SST_K = T0+1                # Assume a 1K air-to-sea temperature difference
es = thermo.calc_es(SST_K - 273.15)
qs = 0.622 * es / (p0 - es) # saturation specific humidity

# Calculate the density temperature of the parcel
Trho_lift_rev_s, Trho_lift_pa_s = density_temp(SST_K, qs, p0, plift)

# Calculate the density temperature of the lifted parcel
# from the boundary layer (first sounding level)
Trho_lift_rev_b, Trho_lift_pa_b = density_temp(T0, q0, p0, plift)

# Calculate the density temperature of the environmental field.
Trho_env = T * (1 + q / 0.622) / (1 + q)         # density temperature, K

# Define the buoyancy as the difference between the density temperature
# of the lifted parcel and the density temperature of the environment.
buoyancy_s_rev = Trho_lift_rev_s - Trho_env[ibest:]
buoyancy_s_pa = Trho_lift_pa_s - Trho_env[ibest:]
buoyancy_b_rev = Trho_lift_rev_b - Trho_env[ibest:]
buoyancy_b_pa = Trho_lift_pa_b - Trho_env[ibest:]

# Calculate CAPE
B_s_rev = np.expand_dims(buoyancy_s_rev, axis = 0)
B_b_rev = np.expand_dims(buoyancy_b_rev, axis = 0)
CAPEs_rev, _ = calculate_CAPE_CIN(p, B_s_rev, 2)
CAPEb_rev, _ = calculate_CAPE_CIN(p, B_b_rev, 2)

# Calculate potential intensity
CAPEs_rev = np.maximum(CAPEs_rev,0)
CAPEs_rev[np.isnan(CAPEs_rev)] = 0
CAPEb_rev = np.maximum(CAPEb_rev,0)
CAPEb_rev[np.isnan(CAPEb_rev)] = 0
cape_diff = (CAPEs_rev-CAPEb_rev)[0]

# Ratio of Ck to Cd
CkCd = 1

# Outflow temperature is defined as the temperature of the
# environment at the level of neutral buoyancy (LNB) for
# the lifted saturated parcel.
idx_outflow = np.argwhere(B_s_rev[0,:]<0).flatten()[0]
T_out = T[idx_outflow-1]

# Potential intensity formula
Vp = (CkCd*SST_K/T_out*cape_diff)**0.5
print('Potential intensity: %.2f m/s' % Vp)

In [ ]:
from importlib import reload
reload(thermo)

# Plot the skew-T log p diagram
thermo.skewtv(p, T_C, rh);
titstring = f"Station {station_id}, {station_name}, {year}/{month}/{day}/{hour}"

# Plot the density temperature of a parcel lifted both reversibly
# and pseudo-adiabatically, alongside the environmental fields.
# Use a log-pressure adjustment (-30 * log(p)) to transform
# temperatures into Skew-T X-axis coordinates.
Tz_env = (Trho_env - 273.15) - 30.0 * np.log(0.001 * p)
Tz_s = (Trho_lift_rev_s - 273.15) - 30.0 * np.log(0.001 * plift)
g1, = plt.plot(Tz_s, plift, 'b-', linewidth=2., label='Ocean')
Tz_b = (Trho_lift_rev_b - 273.15) - 30.0 * np.log(0.001 * plift)
g2, = plt.plot(Tz_b, plift, 'k-', linewidth=2., label='Surface')

g3 = plt.fill_betweenx(plift, Tz_s, Tz_env[ibest:],
                 where=(Tz_s >= Tz_env[ibest:]),
                 interpolate = True, color='blue',
                 alpha=0.2, label=r'$CAPE_s$')
g4 = plt.fill_betweenx(plift, Tz_b, Tz_env[ibest:],
                 where=(Tz_b >= Tz_env[ibest:]),
                 interpolate = True, color='black',
                 alpha=0.2, label=r'$CAPE_b$')

plt.legend(handles=[g1, g2, g3, g4])
plt.title(f"{titstring}; Parcel lifted from {plift[0]:.1f} hPa")
plt.show()